# AI-Powered Smart Retail & Customer Intelligence Platform

## Module 5: FastAPI Backend Integration

This module integrates all trained AI models into a single REST API using FastAPI.

The API provides endpoints for:
- Customer face recognition
- Product image classification
- Customer sentiment analysis
- FAQ chatbot
- Dashboard statistics

The trained models are loaded once during application startup and reused for predictions.

In [1]:
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio

In [2]:
!apt-get update -qq
!apt-get install -y cmake libopenblas-dev liblapack-dev libx11-dev libgtk-3-dev

!pip install face_recognition
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
liblapack-dev is already the newest version (3.10.0-2ubuntu1).
libopenblas-dev is already the newest version (0.3.20+ds-1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
libgtk-3-dev is already the newest version (3.24.33-1ubuntu2.2).
libx11-dev is already the newest version (2:1.7.5-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 130 not upgraded.


In [3]:
import face_recognition
print("face_recognition version:", face_recognition.__version__)

face_recognition version: 1.2.3


In [4]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "face_db.pkl" in files:
        print("Found:", os.path.join(root, "face_db.pkl"))

Found: /content/drive/MyDrive/Smart-Retail-AI/models/face_db.pkl


In [5]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Resume (4).pdf', '23BCE11316.pdf', 'Resume (3).pdf', 'Ganga Bhumi Club.pdf', 'Maths assignment.pdf', 'Document from Ishita Gautam', 'RESUME (2) (1).pdf', 'RESUME (2)-1.pdf', 'RESUME (2).pdf', 'Ishita Gautam - offer letter.pdf', 'Offer letter acceptance pdf.pdf', 'DOC-20250521-WA0015..pdf', 'Mini task 2.pdf', 'Installation .png', 'Hackhathon.png', 'wchl.png', 'Resume (2).pdf', 'Resume (1).pdf', 'StudentGradeHistory_23BCE11316_page-0002 (1).jpg', 'StudentGradeHistory_23BCE11316_page-0001.jpg', 'StudentGradeHistory_23BCE11316_page-0002.jpg', 'WhatsApp Image 2025-07-23 at 22.46.52_87fc5eb0.jpg', 'back.jpg', 'Front .jpg', '39.pdf', 'BlockseBlock internship certificate.pdf', 'question-2276389 (2).pdf', 'question-2276389 (1).pdf', 'question-2276389.pdf', 'question-2276389 (1).gdoc', 'question-2276389.gdoc', 'class 12th.pdf ..pdf', 'class11th.pdf', 'class 8th.pdf', 'Class 8th .pdf', 'Test Preparation Documents - to be shared with test takers.zip', 'Python Screening Task 2 .pdf', 'class 12th.

In [7]:
print(os.listdir("/content/drive/MyDrive/Smart-Retail-AI"))

['notebooks', 'datasets', 'models', 'outputs', 'images']


In [8]:
import os

model_path = "/content/drive/MyDrive/Smart-Retail-AI/models"

print(os.listdir(model_path))

['product_classifier.keras', 'face_db.pkl', 'sentiment_model.pkl', 'vectorizer.pkl', 'chatbot_model.pkl', 'chatbot_vectorizer.pkl', 'label_encoder.pkl']


In [9]:
dataset_path = "/content/drive/MyDrive/Smart-Retail-AI/datasets"

print(os.listdir(dataset_path))

['Womens Clothing E-Commerce Reviews.csv', 'product_images', 'customer_faces', 'lfw-deepfunneled', 'intents.json']


In [10]:
import pickle

MODEL_PATH = "/content/drive/MyDrive/Smart-Retail-AI/models"

with open(f"{MODEL_PATH}/face_db.pkl", "rb") as f:
    face_database = pickle.load(f)

print("Face Database Loaded")
print("Total Customers:", len(face_database))

first_customer = list(face_database.keys())[0]

print("Example Customer:", first_customer)
print("Name:", face_database[first_customer]["name"])
print("Number of Encodings:", len(face_database[first_customer]["encodings"]))

Face Database Loaded
Total Customers: 30
Example Customer: Customer_001
Name: George_W_Bush
Number of Encodings: 5


In [11]:
for customer_id, data in face_database.items():
    if "Serena" in data["name"]:
        print(customer_id, data["name"])

Customer_012 Serena_Williams


In [12]:
import io
import threading
import pickle

import face_recognition

from fastapi import FastAPI, UploadFile, File
import uvicorn
import nest_asyncio

In [13]:
app = FastAPI(
    title="Smart Retail AI API"
)

print("FastAPI created")

FastAPI created


In [14]:
@app.get("/")
def home():
    return {
        "message": "Smart Retail AI API is running successfully"
    }

print("Home API added")

Home API added


In [15]:
@app.post("/recognize-face")
async def recognize_face(file: UploadFile = File(...)):

    image_bytes = await file.read()

    image = face_recognition.load_image_file(
        io.BytesIO(image_bytes)
    )


    uploaded_faces = face_recognition.face_encodings(image)


    if len(uploaded_faces) == 0:
        return {
            "message": "No face detected"
        }


    uploaded_encoding = uploaded_faces[0]


    for customer_id, data in face_database.items():

        matches = face_recognition.compare_faces(
            data["encodings"],
            uploaded_encoding,
            tolerance=0.6
        )


        if True in matches:

            return {
                "customer_id": customer_id,
                "name": data["name"],
                "status": "Returning customer"
            }


    return {
        "customer_id": None,
        "name": "Unknown",
        "status": "New customer"
    }


print("Face API added")

Face API added


In [16]:
nest_asyncio.apply()


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


print("Server ready")

Server ready


In [17]:
thread = threading.Thread(
    target=run_server
)

thread.start()

In [18]:
from pyngrok import ngrok

ngrok.set_auth_token("3HLi4qyM5u9IDFhjVXeUp0bXz0h_3EYsuCXuK2XJSJZV5aRsF")

print("Ngrok authenticated")

INFO:     Started server process [32943]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


Ngrok authenticated


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [19]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://specks-subject-probation.ngrok-free.dev" -> "http://localhost:8000"


In [20]:
import pickle

MODEL_PATH = "/content/drive/MyDrive/Smart-Retail-AI/models"


with open(f"{MODEL_PATH}/sentiment_model.pkl", "rb") as f:
    sentiment_model = pickle.load(f)


with open(f"{MODEL_PATH}/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)


print("Sentiment model loaded")

Sentiment model loaded


In [21]:
from pydantic import BaseModel


class TextRequest(BaseModel):
    text: str

In [22]:
@app.post("/analyze-sentiment")
def analyze_sentiment(request: TextRequest):

    text = request.text


    transformed_text = vectorizer.transform(
        [text]
    )


    prediction = sentiment_model.predict(
        transformed_text
    )


    return {
        "text": text,
        "sentiment": str(prediction[0])
    }

In [23]:
thread = threading.Thread(
    target=run_server
)

thread.start()

INFO:     Started server process [32943]


In [24]:
from pyngrok import ngrok

ngrok.kill()

print("Cleaned old servers")

INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Cleaned old servers


In [25]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.5Gi       8.6Gi       4.0Mi       2.6Gi        10Gi
Swap:             0B          0B          0B


In [26]:
!lsof -i :8000

COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 32943 root   57u  IPv4 691391      0t0  TCP *:8000 (LISTEN)


In [27]:
import threading
import uvicorn

In [28]:
import io
import threading
import pickle

import face_recognition

from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel

import uvicorn
import nest_asyncio

In [29]:
MODEL_PATH = "/content/drive/MyDrive/Smart-Retail-AI/models"


# Face database
with open(f"{MODEL_PATH}/face_db.pkl", "rb") as f:
    face_database = pickle.load(f)


# Sentiment model
with open(f"{MODEL_PATH}/sentiment_model.pkl", "rb") as f:
    sentiment_model = pickle.load(f)


with open(f"{MODEL_PATH}/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)


print("All models loaded")

All models loaded


In [30]:
app = FastAPI(
    title="Smart Retail AI API"
)

print("App created")

App created


In [31]:
@app.post("/recognize-face")
async def recognize_face(file: UploadFile = File(...)):

    image_bytes = await file.read()

    image = face_recognition.load_image_file(
        io.BytesIO(image_bytes)
    )

    faces = face_recognition.face_encodings(image)

    if len(faces) == 0:
        return {
            "message": "No face detected"
        }


    uploaded_encoding = faces[0]


    for customer_id, data in face_database.items():

        matches = face_recognition.compare_faces(
            data["encodings"],
            uploaded_encoding
        )

        if True in matches:
            return {
                "customer_id": customer_id,
                "name": data["name"],
                "status": "Returning customer"
            }


    return {
        "customer_id": None,
        "name": "Unknown",
        "status": "New customer"
    }

In [32]:
class TextRequest(BaseModel):
    text: str


@app.post("/analyze-sentiment")
def analyze_sentiment(request: TextRequest):

    text = request.text

    transformed = vectorizer.transform(
        [text]
    )

    prediction = sentiment_model.predict(
        transformed
    )

    return {
        "text": text,
        "sentiment": str(prediction[0])
    }

In [33]:
nest_asyncio.apply()


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


thread = threading.Thread(
    target=run_server
)

thread.start()

INFO:     Started server process [32943]


In [34]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


NgrokTunnel: "https://specks-subject-probation.ngrok-free.dev" -> "http://localhost:8000"


In [35]:
CHATBOT_MODEL_PATH = "/content/drive/MyDrive/Smart-Retail-AI/models"


with open(f"{CHATBOT_MODEL_PATH}/chatbot_model.pkl", "rb") as f:
    chatbot_model = pickle.load(f)


with open(f"{CHATBOT_MODEL_PATH}/chatbot_vectorizer.pkl", "rb") as f:
    chatbot_vectorizer = pickle.load(f)


with open(f"{CHATBOT_MODEL_PATH}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)


print("Chatbot models loaded")

Chatbot models loaded


In [36]:
class ChatRequest(BaseModel):
    message: str

In [37]:
@app.post("/chatbot")
def chatbot(request: ChatRequest):

    message = request.message


    transformed_message = chatbot_vectorizer.transform(
        [message]
    )


    prediction = chatbot_model.predict(
        transformed_message
    )


    intent = label_encoder.inverse_transform(
        prediction
    )[0]


    return {
        "message": message,
        "intent": intent
    }

In [38]:
!lsof -i :8000

COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 32943 root   57u  IPv4 691391      0t0  TCP *:8000 (LISTEN)


In [39]:
print(chatbot_model)

LogisticRegression(max_iter=1000)


In [40]:
print(app.routes)

[Route(path='/openapi.json', name='openapi', methods=['GET', 'HEAD']), Route(path='/docs', name='swagger_ui_html', methods=['GET', 'HEAD']), Route(path='/docs/oauth2-redirect', name='swagger_ui_redirect', methods=['GET', 'HEAD']), Route(path='/redoc', name='redoc_html', methods=['GET', 'HEAD']), APIRoute(path='/recognize-face', name='recognize_face', methods=['POST']), APIRoute(path='/analyze-sentiment', name='analyze_sentiment', methods=['POST']), APIRoute(path='/chatbot', name='chatbot', methods=['POST'])]


In [41]:
thread = threading.Thread(
    target=run_server
)

thread.start()

INFO:     Started server process [32943]
INFO:     Waiting for application startup.


In [42]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8000)

print(public_url)

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


NgrokTunnel: "https://specks-subject-probation.ngrok-free.dev" -> "http://localhost:8000"


In [43]:
import pickle

MODEL_PATH = "/content/drive/MyDrive/Smart-Retail-AI/models"

with open(f"{MODEL_PATH}/chatbot_model.pkl", "rb") as f:
    chatbot_model = pickle.load(f)

with open(f"{MODEL_PATH}/chatbot_vectorizer.pkl", "rb") as f:
    chatbot_vectorizer = pickle.load(f)

with open(f"{MODEL_PATH}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

print("Chatbot models loaded")

Chatbot models loaded


In [44]:
class ChatRequest(BaseModel):
    message: str

In [45]:
@app.post("/chatbot")
def chatbot(request: ChatRequest):

    message = request.message

    transformed_message = chatbot_vectorizer.transform(
        [message]
    )

    prediction = chatbot_model.predict(
        transformed_message
    )

    intent = label_encoder.inverse_transform(
        prediction
    )[0]

    return {
        "message": message,
        "intent": intent
    }

print("Chatbot API added")

Chatbot API added


In [46]:
for route in app.routes:
    print(route.path)

/openapi.json
/docs
/docs/oauth2-redirect
/redoc
/recognize-face
/analyze-sentiment
/chatbot
/chatbot


In [47]:
thread = threading.Thread(
    target=run_server
)

thread.start()

INFO:     Started server process [32943]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.


In [48]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8000)

print(public_url)

INFO:     Application shutdown complete.


NgrokTunnel: "https://specks-subject-probation.ngrok-free.dev" -> "http://localhost:8000"


In [49]:
for route in app.routes:
    print(route.path)

/openapi.json
/docs
/docs/oauth2-redirect
/redoc
/recognize-face
/analyze-sentiment
/chatbot
/chatbot


In [50]:
def run_server_new():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8001
    )


thread = threading.Thread(
    target=run_server_new
)

thread.start()

INFO:     Started server process [32943]
INFO:     Waiting for application startup.


In [52]:
for route in app.routes:
    print(route.path)

/openapi.json
/docs
/docs/oauth2-redirect
/redoc
/recognize-face
/analyze-sentiment
/chatbot
/chatbot


In [54]:
def run_server_new():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8001
    )


thread = threading.Thread(
    target=run_server_new
)

thread.start()

INFO:     Started server process [32943]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [57]:
from pyngrok import ngrok

public_url = ngrok.connect(8001)

print(public_url)

NgrokTunnel: "https://specks-subject-probation.ngrok-free.dev" -> "http://localhost:8001"
